In [ ]:
from unittest import result

import pandas as pd
import numpy as np
import re
import unicodedata
from narwhals import group_by

from pandas.core.computation.parsing import tokenize_string

FILENAME = "F:/Milestone 2/sampleddata.csv"

In [ ]:

df = pd.read_csv(FILENAME)

df

In [ ]:
df = df.replace(r'\xa0', ' ', regex=True)
en = df[df['lang'] == 'en']
en

In [ ]:
def clean_tweet(df):
    df['cleanedTweet'] = df['rawContent'].str.normalize('NFKD')
    df['cleanedTweet'] = df['cleanedTweet'].astype(str).str.replace(r'(\n)', lambda m: "",regex=True).str.strip()
    df['cleanedTweet'] = df['cleanedTweet'].str.replace(r"^.{0,5}(@\S*\b\s)+|(https:\S*\b)",lambda m: "",regex=True)
    df['cleanedTweet'] = df['cleanedTweet'].str.replace(r'\s+', ' ', regex=True)
    return df

clean_tweet(en)

In [ ]:
explosion = en.explode('mentionedUsers').reset_index(drop=True)
dict_series = explosion['mentionedUsers'].apply(lambda x: x if isinstance(x, list) else {})
normed = pd.json_normalize(dict_series)
#result = pd.concat([explosion['id'], normed[['name','screen_name']]], axis=1)
#result
explosion

In [ ]:
import ast
#en['mentionedUsers'] = en['mentionedUsers'].apply(lambda x: ast.literal_eval(x))
# 1. Explode the list (missing rows are safely kept as NaN)
exploded_df = en.explode('mentionedUsers').reset_index(drop=True)

# 2. Fill NaN rows with an empty dict so json_normalize doesn't fail
dict_series = exploded_df['mentionedUsers'].apply(lambda x: x if isinstance(x, dict) else {})

# 3. Normalize the dictionaries into columns
normalized_cols = pd.json_normalize(dict_series)

# 4. Combine the ID with the extracted values
result = pd.concat([exploded_df['id'], normalized_cols[['name', 'screen_name']]], axis=1)

users = result.groupby(['screen_name','name']).agg(count =('screen_name','size'))
users

In [ ]:
pd.read_csv("../DataFiles/combined_output.csv")

In [ ]:
pd.read_csv("../fixed/octobergap_chunk_14.csv")

In [ ]:
pd.read_csv("../x-24-us-election/part_47/octobergap_chunk_41.csv.gz")

In [ ]:
df[df['retweetedTweet'] == True]

In [ ]:
octber = pd.read_csv('../x-24-us-election/part_19/may_july_chunk_380.csv.gz', low_memory=False)


In [ ]:
octber['date'] = pd.to_datetime(octber['date'])
min_rows = octber[octber['date'] == octber['date'].min()]

In [ ]:
min_rows

In [ ]:
octber

In [ ]:
octber[octber['retweetedTweet'] == '1811339675384357201']

In [ ]:
df2 = pd.read_csv("F:/Milestone 2/raw_sample_data.csv")
df2['date']

In [ ]:
df3 = pd.read_csv("F:/Milestone 2/sampleddata.csv")
combined = df3.groupby("week").agg(
    min_date = ('date', 'min'),
    max_date = ('date', 'max')
)
date_cols = ['min_date', 'max_date']
combined[date_cols] = combined[date_cols].apply(pd.to_datetime)
combined[date_cols] = combined[date_cols].apply(lambda x: x.dt.strftime('%B %d'))
combined['week_range'] = combined['min_date'].astype(str).str.cat(combined['max_date'].astype(str), sep=' - ', na_rep='')
combined.to_csv('weeks_table.csv')

In [ ]:
pd.set_option('display.float_format', lambda x: f'{x:.1f}')

df4 = pd.read_csv("F:/Milestone 2/sampleddata.csv")
#df4 = df4.drop_duplicates(subset='id', keep='first')
#df4.to_csv("F:/Milestone 2/sampleddata.csv")
df4[df4['rawContent'].str.startswith('RT@', na=False)]